# NexusCommerce — Train Base Models on a Custom CSV

Use this notebook when you have **your own CSV** instead of the Kaggle competition dataset.

### Steps
1. Upload your CSV via **Add Data → Upload** in Kaggle (or attach a dataset).
2. Set `CSV_PATH` and `COLUMN_MAP` in **Cell 3** to match your file.
3. Run all cells.
4. Download the model file from the **Output** tab.

### Minimum CSV requirements
Your CSV must have **at least** a date column and a quantity/sales column.

| Required | Accepted column names |
|----------|-----------------------|
| Date | `date`, `sale_date`, `order_date`, `Day` |
| Quantity | `sales`, `quantity`, `qty`, `units_sold`, `demand` |

In [ ]:
# No extra installs needed — scikit-learn is pre-installed on Kaggle

In [ ]:
import os, pickle, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("All imports OK")

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CONFIGURE THIS CELL FOR YOUR CSV                       ║
# ╚══════════════════════════════════════════════════════════╝

# Path to your uploaded CSV
CSV_PATH = '/kaggle/input/your-dataset/your_file.csv'   # ← change this

# Map YOUR column names → internal names (leave value as None to auto-detect)
COLUMN_MAP = {
    'date_col':     None,   # e.g. 'Order Date' or 'date'
    'quantity_col': None,   # e.g. 'Quantity' or 'sales'
    # Optional — set to None if not present
    'product_col':  None,   # e.g. 'Product Name'
    'revenue_col':  None,   # e.g. 'Revenue'
}

# Set True if your dates need a custom format (e.g. '15/08/2020')
DATE_FORMAT = None   # e.g. '%d/%m/%Y'

OUTPUT_DIR  = '/kaggle/working/models/base'
TRAIN_RATIO = 0.80
RANDOM_SEED = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output: {OUTPUT_DIR}')

---
## 1 — Load & Auto-detect Columns

In [ ]:
raw = pd.read_csv(CSV_PATH)
print(f'Shape   : {raw.shape}')
print(f'Columns : {list(raw.columns)}')
raw.head(3)

In [ ]:
def _detect(df, candidates):
    cols_lower = {c.lower().replace(' ', '_'): c for c in df.columns}
    for c in candidates:
        if c.lower().replace(' ', '_') in cols_lower:
            return cols_lower[c.lower().replace(' ', '_')]
    return None

# Auto-detect columns if not set manually
date_col = COLUMN_MAP['date_col'] or _detect(
    raw, ['date', 'sale_date', 'order_date', 'Day', 'Date'])
qty_col  = COLUMN_MAP['quantity_col'] or _detect(
    raw, ['sales', 'quantity', 'qty', 'units_sold', 'demand', 'Sales', 'Quantity'])

if date_col is None:
    raise ValueError(
        'Could not detect a date column. Set COLUMN_MAP["date_col"] manually.')
if qty_col is None:
    raise ValueError(
        'Could not detect a quantity column. Set COLUMN_MAP["quantity_col"] manually.')

print(f'Date column     : "{date_col}"')
print(f'Quantity column : "{qty_col}"')

df = raw[[date_col, qty_col]].copy()
df.columns = ['sale_date', 'quantity']
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce').fillna(0)

if DATE_FORMAT:
    df['sale_date'] = pd.to_datetime(df['sale_date'], format=DATE_FORMAT)
else:
    df['sale_date'] = pd.to_datetime(df['sale_date'], infer_datetime_format=True)

print(f'\nDate range : {df["sale_date"].min().date()}  →  {df["sale_date"].max().date()}')
print(f'Total rows : {len(df):,}')
df.head(3)

---
## 2 — Preprocessing & Feature Engineering

In [ ]:
# Aggregate to daily totals
daily = (
    df.groupby('sale_date')['quantity']
    .sum()
    .reset_index()
    .set_index('sale_date')
    .asfreq('D')
    .ffill()
    .fillna(0)
)
daily.columns = ['quantity']

print(f'Daily rows : {len(daily)}')
if len(daily) < 60:
    print('⚠️  WARNING: fewer than 60 days of data. Models may be unreliable.')

# Plot
plt.figure(figsize=(13, 3))
plt.plot(daily.index, daily['quantity'], linewidth=0.9, color='steelblue')
plt.title('Daily Sales — your dataset'); plt.ylabel('Quantity'); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
FEATURE_COLS = ['month', 'day', 'day_of_week', 'product_id',
                'lag_7', 'lag_14', 'rolling_mean_7']
TARGET_COL   = 'quantity'

f = daily.copy()
f['month']          = f.index.month
f['day']            = f.index.day
f['day_of_week']    = f.index.dayofweek
f['product_id']     = 0
f['lag_7']          = f['quantity'].shift(7)
f['lag_14']         = f['quantity'].shift(14)
f['rolling_mean_7'] = f['quantity'].rolling(7).mean()
features = f.dropna()

split   = int(len(features) * TRAIN_RATIO)
train   = features.iloc[:split]
test    = features.iloc[split:]
X_train = train[FEATURE_COLS].values;  y_train = train[TARGET_COL].values
X_test  = test[FEATURE_COLS].values;   y_test  = test[TARGET_COL].values

print(f'Train : {len(train):,}  |  Test : {len(test):,}')

In [ ]:
def metrics(y_true, y_pred, name):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mask = y_true != 0
    mape = float(np.mean(np.abs((y_true[mask]-y_pred[mask])/y_true[mask]))*100) if mask.sum() else 0.0
    print(f'\n{name}: MAE={mae:.2f}  RMSE={rmse:.2f}  R²={r2:.4f}  MAPE={mape:.2f}%')
    return {'model': name, 'mae': mae, 'rmse': rmse, 'r2_score': r2, 'mape': mape}

all_m = []

---
## 3 — Train Random Forest

In [ ]:
print('Training Random Forest...')
rf = RandomForestRegressor(
    n_estimators=200, max_depth=15,
    min_samples_split=5, min_samples_leaf=2,
    random_state=RANDOM_SEED, n_jobs=-1)
rf.fit(X_train, y_train)
all_m.append(metrics(y_test, rf.predict(X_test), 'Random Forest'))
with open(os.path.join(OUTPUT_DIR,'rf_model.pkl'),'wb') as fh: pickle.dump(rf,fh)
print('✅ rf_model.pkl saved')

---
## 4 — Metrics & Output

In [ ]:
cmp = pd.DataFrame(all_m).set_index("model").round(4)
print("\n" + "="*55)
print("  RANDOM FOREST METRICS")
print("="*55)
print(cmp.to_string())
cmp.to_csv("/kaggle/working/model_metrics.csv")

print(f"\nFiles in {OUTPUT_DIR}:")
for fn in os.listdir(OUTPUT_DIR):
    sz = os.path.getsize(os.path.join(OUTPUT_DIR, fn))
    print(f"  {fn:<25} {sz/1024:.1f} KB")

print("\nDownload rf_model.pkl -> copy to nexuscommerce/ml/models/base/")